In [20]:
import os
import csv
import warnings
import ast 
import itertools
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import urllib.request
import joblib
import math

from sklearn.ensemble import RandomForestRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, RepeatedKFold, cross_val_predict, LeaveOneOut, cross_validate, LearningCurveDisplay, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, make_scorer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, average_precision_score
from matplotlib.colors import ListedColormap 
from scipy.stats import spearmanr
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import PowerTransformer
from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin, clone
from scipy.optimize import minimize_scalar
from scipy.stats import yeojohnson

# ==========================================
# 🌟 NEW: ANTIBERTY & ABLANG2 DEPENDENCY CHECKS
# ==========================================
try:
    from antiberty import AntiBERTyRunner
    ANTIBERTY_AVAILABLE = True
except ImportError:
    ANTIBERTY_AVAILABLE = False

try:
    import ablang2
    ABLANG2_AVAILABLE = True
except ImportError:
    ABLANG2_AVAILABLE = False

# ==========================================
# 🌟 GEORGIEV FEATURES DICTIONARY
# ==========================================
GEORGIEV_DICT = {
    'A': [0.57, 3.37, -3.66, 2.34, -1.07, -0.4, 1.23, -2.32, -2.01, 1.31, -1.14, 0.19, 1.66, 4.39, 0.18, -2.6, 1.49, 0.46, -4.22],
    'C': [2.66, -1.52, -3.29, -3.77, 2.96, -2.23, 0.44, -3.49, 2.22, -3.78, 1.98, -0.43, -1.03, 0.93, 1.43, 1.45, -1.15, -1.64, -1.05],
    'D': [-2.46, -0.66, -0.57, 0.14, 0.75, 0.24, -5.15, -1.17, 0.73, 1.5, 1.51, 5.61, -3.85, 1.28, -1.98, 0.05, 0.9, 1.38, -0.03],
    'E': [-3.08, 3.45, 0.05, 0.62, -0.49, 0, -5.66, -0.11, 1.49, -2.26, -1.62, -3.97, 2.3, -0.06, -0.35, 1.51, -2.29, -1.47, 0.15],
    'F': [3.12, 0.68, 2.4, -0.35, -0.88, 1.62, -0.15, -0.41, 4.2, 0.73, -0.56, 3.54, 5.25, 1.73, 2.14, 1.1, 0.68, 1.46, 2.33],
    'G': [0.15, -3.49, -2.97, 2.06, 0.7, 7.47, 0.41, 1.62, -0.47, -2.9, -0.98, -0.62, -0.11, 0.15, -0.53, 0.35, 0.3, 0.32, 0.05],
    'H': [-0.39, 1, -0.63, -3.49, 0.05, 0.41, 1.61, -0.6, 3.55, 1.52, -2.28, -3.12, -1.45, -0.77, -4.18, -2.91, 3.37, 1.87, 2.17],
    'I': [3.1, 0.37, 0.26, 1.04, -0.05, -1.18, -0.21, 3.45, 0.86, 1.98, 0.89, -1.67, -1.02, -1.21, -1.78, 5.71, 1.54, 2.11, -4.18],
    'K': [-3.89, 1.47, 1.95, 1.17, 0.53, 0.1, 4.01, -0.01, -0.26, -1.66, 5.86, -0.06, 1.38, 1.78, -2.71, 1.62, 0.96, -1.09, 1.36],
    'L': [2.72, 1.88, 1.92, 5.33, 0.08, 0.09, 0.27, -4.06, 0.43, -1.2, 0.67, -0.29, -2.47, -4.79, 0.8, -1.43, 0.63, -0.24, 1.01],
    'M': [1.89, 3.88, -1.57, -3.58, -2.55, 2.07, 0.84, 1.85, -2.05, 0.78, 1.53, 2.44, -0.26, -3.09, -1.39, -1.02, -4.32, -1.34, 0.09],
    'N': [-2.02, -1.92, 0.04, -0.65, 1.61, 2.08, 0.4, -2.47, -0.07, 7.02, 1.32, -2.44, 0.37, -0.89, 3.13, 0.79, -1.54, -1.71, -0.25],
    'P': [-0.58, -4.33, -0.02, -0.21, -8.31, -1.82, -0.12, -1.18, 0, -0.66, 0.64, -0.92, -0.37, 0.17, 0.36, 0.08, 0.16, -0.34, 0.04],
    'Q': [-2.54, 1.82, -0.82, -1.85, 0.09, 0.6, 0.25, 2.11, -1.92, -1.67, 0.7, -0.27, -0.99, -1.56, 6.22, -0.18, 2.72, 4.35, 0.92],
    'R': [-2.8, 0.31, 2.84, 0.25, 0.2, -0.37, 3.81, 0.98, 2.43, -0.99, -4.9, 2.09, -3.08, 0.82, 1.32, 0.69, -2.62, -1.49, -2.57],
    'S': [-1.1, -2.05, -2.19, 1.36, 1.78, -3.36, 1.39, -1.21, -2.83, 0.39, -2.92, 1.27, 2.86, -1.88, -2.42, 1.75, -2.77, 3.36, 2.67],
    'T': [-0.65, -1.6, -1.39, 0.63, 1.35, -2.45, -0.65, 3.43, 0.34, 0.24, -0.53, 1.91, 2.66, -3.07, 0.2, -2.2, 3.73, -5.46, -0.73],
    'V': [2.64, 0.03, -0.67, 2.34, 0.64, -2.01, -0.33, 3.93, -0.21, 1.27, 0.43, -1.71, -2.93, 4.22, 1.06, -1.31, -1.97, -1.21, 4.77],
    'W': [1.89, -0.09, 4.21, -2.77, 0.72, 0.86, -1.07, -1.66, -5.87, -0.66, -2.49, -0.3, -0.5, 1.64, -0.72, 1.75, 2.73, -2.2, 0.9],
    'Y': [0.79, -2.62, 4.11, -0.63, 1.89, -0.53, -1.3, 1.31, -0.56, -0.95, 1.91, -1.26, 1.57, 0.2, -0.76, -5.19, -2.56, 2.87, -3.43]
}

# --- SILENCE WARNINGS ---
def completely_silence_warnings(*args, **kwargs):
    pass
warnings.warn = completely_silence_warnings
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings('ignore')

ESM_AVAILABLE = True
PROPERMAB_AVAILABLE = False
sns.set_theme(style="whitegrid")

def custom_spearman(y_true, y_pred):
    sc, _ = spearmanr(y_true, y_pred)
    return sc if not np.isnan(sc) else 0

def calculate_weighted_yeojohnson_lambda(y, weights):
    y_flat = np.asarray(y).flatten()
    w = np.asarray(weights).flatten()
    w = (w / np.sum(w)) * len(w)
    
    def weighted_nll(lmbda):
        y_trans = yeojohnson(y_flat, lmbda)
        w_mean = np.average(y_trans, weights=w)
        w_var = np.average((y_trans - w_mean)**2, weights=w)
        if w_var <= 0: return np.inf
        jacobian = np.sum(w * np.sign(y_flat) * np.log1p(np.abs(y_flat)))
        nll = (len(w) / 2.0) * np.log(w_var) - (lmbda - 1) * jacobian
        return nll
        
    res = minimize_scalar(weighted_nll, bounds=(-3.0, 3.0), method='bounded')
    return res.x

class CustomYeoJohnsonTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, lmbda=1.0):
        self.lmbda = lmbda
    def fit(self, X, y=None): return self
    def transform(self, X):
        X_flat = np.asarray(X).flatten()
        return yeojohnson(X_flat, self.lmbda).reshape(-1, 1)
    def inverse_transform(self, X_trans):
        X_trans_flat = np.asarray(X_trans).flatten()
        X_inv = np.zeros_like(X_trans_flat)
        pos, neg = X_trans_flat >= 0, ~(X_trans_flat >= 0)
        
        if abs(self.lmbda) < 1e-10: X_inv[pos] = np.exp(X_trans_flat[pos]) - 1
        else: X_inv[pos] = np.maximum(X_trans_flat[pos] * self.lmbda + 1, 0) ** (1 / self.lmbda) - 1
            
        if abs(self.lmbda - 2.0) < 1e-10: X_inv[neg] = 1 - np.exp(-X_trans_flat[neg])
        else: X_inv[neg] = 1 - np.maximum(1 - (2 - self.lmbda) * X_trans_flat[neg], 0) ** (1 / (2 - self.lmbda))
        return X_inv.reshape(-1, 1)

class SelfContainedTargetTransformRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, regressor, transform_type=None, weight_col=None):
        self.regressor = regressor
        self.transform_type = transform_type
        self.weight_col = weight_col
        
    def fit(self, X, y):
        self.regressor_ = clone(self.regressor)
        if self.weight_col:
            if isinstance(X, pd.DataFrame):
                weights = X[self.weight_col].values
                X_clean = X.drop(columns=[self.weight_col])
            else:
                weights = X[:, -1]
                X_clean = X[:, :-1]
        else:
            X_clean = X.copy() if isinstance(X, pd.DataFrame) else np.copy(X)
            weights = None
            
        if self.transform_type == 'log1p': y_trans = np.log1p(y)
        elif self.transform_type in ['box-cox', 'yeo-johnson']:
            self.pt_ = PowerTransformer(method=self.transform_type)
            y_trans = self.pt_.fit_transform(np.asarray(y).reshape(-1, 1)).flatten()
        elif self.transform_type == 'weighted-yeo-johnson':
            if weights is None: weights = np.ones_like(y)
            self.lmbda_ = calculate_weighted_yeojohnson_lambda(y, weights)
            self.pt_ = CustomYeoJohnsonTransformer(lmbda=self.lmbda_)
            y_trans = self.pt_.transform(np.asarray(y).reshape(-1, 1)).flatten()
        else:
            y_trans = y
            
        self.regressor_.fit(X_clean, y_trans)
        return self
        
    def predict(self, X):
        if self.weight_col:
            if isinstance(X, pd.DataFrame): X_clean = X.drop(columns=[self.weight_col])
            else: X_clean = X[:, :-1]
        else:
            X_clean = X
            
        y_pred_trans = self.regressor_.predict(X_clean)
        
        if self.transform_type == 'log1p': return np.expm1(y_pred_trans)
        elif self.transform_type in ['box-cox', 'yeo-johnson']:
            return self.pt_.inverse_transform(y_pred_trans.reshape(-1, 1)).flatten()
        elif self.transform_type == 'weighted-yeo-johnson':
            return self.pt_.inverse_transform(y_pred_trans.reshape(-1, 1)).flatten()
        else:
            return np.asarray(y_pred_trans).flatten()

def load_and_clean_data(filepath, remove_outlier=False):
    print(f"Loading data from {filepath}...")
    df = pd.read_csv(filepath)
    df.columns = df.columns.str.strip()
    
    if remove_outlier and 'Samples' in df.columns:
        initial_len = len(df)
        outliers_to_remove = ['H57L46', 'H57L38']
        df = df[~df['Samples'].str.strip().isin(outliers_to_remove)].reset_index(drop=True)
        rows_dropped = initial_len - len(df)
        if rows_dropped > 0: print(f"  -> SUCCESS: Outliers removed. ({rows_dropped} rows dropped)")
    return df

def filter_redundant_sequences(df, seq_cols):
    parents = set()
    for col_a in seq_cols:
        for col_b in seq_cols:
            if col_a == col_b: continue
            val_a, val_b = df[col_a].dropna(), df[col_b].dropna()
            if val_a.empty or val_b.empty: continue
            str_a = ''.join(str(val_a.iloc[0]).split()).replace(',', '').upper()
            str_b = ''.join(str(val_b.iloc[0]).split()).replace(',', '').upper()
            if len(str_a) > 5 and str_a in str_b and len(str_a) < len(str_b):
                parents.add(col_b)
    return [c for c in seq_cols if c not in parents], list(parents)

def load_aaindex():
    filepath, aaindex_dict, aaindex_desc = 'aaindex1.txt', {}, {}
    if not os.path.exists(filepath):
        try: urllib.request.urlretrieve("https://www.genome.jp/ftp/db/community/aaindex/aaindex1", filepath)
        except Exception: return aaindex_dict, aaindex_desc
            
    try:
        with open(filepath, 'r') as f: lines = f.readlines()
        current_id, current_desc = None, None
        for i in range(len(lines)):
            if lines[i].startswith('H '): current_id = lines[i].split()[1]
            elif lines[i].startswith('D '): current_desc = lines[i][2:].strip()
            elif lines[i].startswith('I '):
                vals1 = [float(x) if x != 'NA' else np.nan for x in lines[i+1].strip().split()]
                vals2 = [float(x) if x != 'NA' else np.nan for x in lines[i+2].strip().split()]
                if len(vals1) == 10 and len(vals2) == 10:
                    aa_keys = list('ARNDCQEGHILKMFPSTWYV')
                    aaindex_dict[current_id] = dict(zip(aa_keys, vals1 + vals2))
                    aaindex_desc[current_id] = current_desc
    except Exception: pass
    return aaindex_dict, aaindex_desc

def extract_sequence_features(df, is_inference=False, dataset_name="default_dataset", esm_model_names=None, cache_tag=""):
    if esm_model_names is None: esm_model_names = ["facebook/esm2_t6_8M_UR50D"]
    
    df_feat = df.copy()
    standard_aas = list('ARNDCQEGHILKMFPSTWYV')
    
    initial_seq_cols = []
    for col in df_feat.columns:
        valid_data = df_feat[col].dropna()
        if not valid_data.empty:
            clean_sample = ''.join(str(valid_data.iloc[0]).split()).replace(',', '').upper()
            if len(clean_sample) > 3 and sum(c in standard_aas for c in clean_sample) / len(clean_sample) > 0.8:
                initial_seq_cols.append(col)

    seq_cols = initial_seq_cols

    if not is_inference:
        print(f"\n==================================================================")
        print(f"🧬 EVALUATING SUBREGION DIVERSITY & SUITABILITY")
        print(f"==================================================================")
        
        valid_seq_cols = []
        for col in seq_cols:
            valid_seqs = df_feat[col].dropna().astype(str).str.replace(r'\s+|,', '', regex=True).str.upper()
            valid_seqs = valid_seqs[~valid_seqs.isin(['NAN', 'NONE', ''])]
            
            if valid_seqs.empty: continue
            
            sample_seq = valid_seqs.iloc[0]
            if len(sample_seq) <= 3:
                print(f"  -> ✂️ DROPPED [{col}]: Too short (<= 3 amino acids).")
                continue
                
            num_unique = valid_seqs.nunique()
            if num_unique <= 1:
                print(f"  -> ❌ DROPPED [{col}]: Constant sequence across all samples (Zero Variance).")
                continue
            elif num_unique <= 4:
                unique_seqs = valid_seqs.unique().tolist()
                print(f"  -> ✅ KEPT [{col}]: Low variance ({num_unique} variants). Sequences: {unique_seqs}")
                valid_seq_cols.append(col)
            else:
                print(f"  -> ✅ KEPT [{col}]: High variance ({num_unique} variants).")
                valid_seq_cols.append(col)
        
        seq_cols, redundant_parents = filter_redundant_sequences(df_feat, valid_seq_cols)
        
        for parent in redundant_parents:
            print(f"  -> ♻️ DROPPED [{parent}]: Redundant parent sequence (contained entirely within another region).")
            
        if 'G4S Linker1_HCK' in seq_cols:
            print("  -> ♻️ Dropping 'G4S Linker1_HCK' (manual override: near-zero variance).")
            seq_cols.remove('G4S Linker1_HCK')
        if 'Media_Type' in seq_cols:
            print("  -> ♻️ Dropping 'Media_Type' (manual override).")
            seq_cols.remove('Media_Type')
        if 'HCH' in seq_cols:
            print("  -> ♻️ Dropping 'HCH' (manual override).")
            seq_cols.remove('HCH')
        if 'Method' in seq_cols:
            print("  -> ♻️ Dropping 'Method' (manual override).")
            seq_cols.remove('Method')
        if 'Manual_Split_Group' in seq_cols:
            print("  -> ♻️ Dropping 'Manual_Split_Group' (manual override).")
            seq_cols.remove('Manual_Split_Group')
        print(f"\nProceeding with {len(seq_cols)} granular building blocks: {seq_cols}\n")
    
    # Global Chains: Direct Extraction
    def extract_direct_seq(row, col_name):
        if col_name in df_feat.columns and pd.notna(row[col_name]):
            val = str(row[col_name]).strip().upper()
            if val not in ['NAN', 'NONE', '']:
                return val
        return 'NAN'

    # Extract directly from the CSV columns
    df_feat['Global_VH'] = df_feat.apply(lambda r: extract_direct_seq(r, 'CD3 VH_HCK'), axis=1)
    df_feat['Global_VL'] = df_feat.apply(lambda r: extract_direct_seq(r, 'CD3 VL_HCK'), axis=1)
    
    def build_fv_with_linker(r):
        if r['Global_VH'] == 'NAN' or r['Global_VL'] == 'NAN': 
            return 'NAN'
        linker = ''
        if 'G4S Linker2_HCK' in df_feat.columns and pd.notna(r['G4S Linker2_HCK']):
            val = str(r['G4S Linker2_HCK']).strip().upper()
            if val not in ['NAN', 'NONE']:
                linker = val
        return r['Global_VH'] + linker + r['Global_VL']
        
    df_feat['Global_Fv'] = df_feat.apply(build_fv_with_linker, axis=1)
    
    all_seq_cols_to_process = seq_cols + ['Global_VH', 'Global_VL', 'Global_Fv']
    aaindex_db, aaindex_desc = load_aaindex()
    
    generated_features = []
    new_columns = {}
    expected_dataset_len = len(df_feat)
    
    cache_dir = os.path.join("feature_cache", f"{dataset_name}_{expected_dataset_len}samples" + (f"_{cache_tag}" if cache_tag else ""))
    os.makedirs(cache_dir, exist_ok=True)
    
    def _load_valid_cache(cache_path, expected_len):
        if os.path.exists(cache_path):
            try:
                with np.load(cache_path) as cached_data:
                    if len(cached_data.files) > 0 and len(cached_data[cached_data.files[0]]) == expected_len:
                        return {k: cached_data[k] for k in cached_data.files}
            except Exception: pass
        return None
        
    print("\n==================================================================")
    print("🧬 STARTING FEATURE EXTRACTION & LOADING")
    print("==================================================================")

    for col in all_seq_cols_to_process:
        print(f"\n⚙️  Processing region: {col}")
        seqs = df_feat[col].astype(str).str.replace(r'\s+|,', '', regex=True).str.upper()

        # AAC
        aac_cache = os.path.join(cache_dir, f"{col}_aac_features_N{expected_dataset_len}.npz")
        cached_dict = _load_valid_cache(aac_cache, expected_dataset_len) if not is_inference else None
        if cached_dict:
            print("   -> [AAC] Loaded from cache")
            for k, v in cached_dict.items(): new_columns[k] = v; generated_features.append(k)
        else:
            print("   -> [AAC] Calculating new features...")
            aac_features = {f'{col}_AAC_{aa}': seqs.apply(lambda x: x.count(aa) if x != 'NAN' else 0).to_numpy() for aa in standard_aas}
            if not is_inference: np.savez(aac_cache, **aac_features)
            for k, v in aac_features.items(): new_columns[k] = v; generated_features.append(k)
                
        # AAindex
        aaindex_cache = os.path.join(cache_dir, f"{col}_aaindex_features_N{expected_dataset_len}.npz")
        cached_dict = _load_valid_cache(aaindex_cache, expected_dataset_len) if not is_inference else None
        if cached_dict:
            print("   -> [AAindex] Loaded from cache")
            for k, v in cached_dict.items(): new_columns[k] = v; generated_features.append(k)
        else:
            print("   -> [AAindex] Calculating new features...")
            aaindex_features = {}
            if aaindex_db:
                for code, prop_map in aaindex_db.items():
                    def calc_prop(seq, pmap=prop_map):
                        if seq == 'NAN' or not seq: return 0
                        vals = [pmap.get(aa) for aa in seq if pmap.get(aa) is not None and not np.isnan(pmap.get(aa))]
                        return sum(vals)/len(vals) if vals else 0
                    aaindex_features[f'{col}_AAindex_{code}'] = seqs.apply(calc_prop).to_numpy()
            if aaindex_features:
                if not is_inference: np.savez(aaindex_cache, **aaindex_features)
                for k, v in aaindex_features.items(): new_columns[k] = v; generated_features.append(k)
                    
        # ESM-2
        if ESM_AVAILABLE:
            for esm_model_name in esm_model_names:
                esm_tag = "ESM_Small_8M" if "8M" in esm_model_name else ("ESM_Medium_35M" if "35M" in esm_model_name else ("ESM_Big_650M" if "650M" in esm_model_name else "ESM_Custom"))
                esm_cache = os.path.join(cache_dir, f"{col}_{esm_tag}_features_N{expected_dataset_len}.npz")
                cached_dict = _load_valid_cache(esm_cache, expected_dataset_len) if not is_inference else None
                if cached_dict:
                    print(f"   -> [{esm_tag}] Loaded from cache")
                    for k, v in cached_dict.items(): new_columns[k] = v; generated_features.append(k)
                else:
                    print(f"   -> [{esm_tag}] Generating neural embeddings (this may take a moment)...")
                    import torch
                    from transformers import EsmModel, EsmTokenizer, logging
                    
                    # SILENCE HUGGING FACE WARNINGS
                    logging.set_verbosity_error()
                    
                    tokenizer = EsmTokenizer.from_pretrained(esm_model_name)
                    model = EsmModel.from_pretrained(esm_model_name)
                    model.eval()
                    
                    embeddings = []
                    for seq in seqs:
                        if seq == 'NAN' or len(seq) < 2: embeddings.append(np.zeros(model.config.hidden_size))
                        else:
                            inputs = tokenizer(seq, return_tensors="pt", add_special_tokens=True)
                            with torch.no_grad():
                                hidden = model(**inputs).last_hidden_state[0]
                                embeddings.append(hidden[1:-1].mean(dim=0).cpu().numpy() if hidden.shape[0] > 2 else hidden.mean(dim=0).cpu().numpy())
                    
                    embeddings = np.array(embeddings)
                    esm_features_dict = {f'{col}_{esm_tag}_{i}': embeddings[:, i] for i in range(embeddings.shape[1])}
                    if not is_inference: np.savez(esm_cache, **esm_features_dict)
                    for k, v in esm_features_dict.items(): new_columns[k] = v; generated_features.append(k)
                    
                # ==========================================
                # 🌟 NEW: ESM 650M SVD COMPRESSION
                # ==========================================
                if esm_tag == "ESM_Big_650M":
                    svd_cache = os.path.join(cache_dir, f"{col}_{esm_tag}_SVD50_features_N{expected_dataset_len}.npz")
                    cached_svd = _load_valid_cache(svd_cache, expected_dataset_len) if not is_inference else None
                    
                    if cached_svd:
                        print(f"   -> [{esm_tag}_SVD50] Loaded from cache")
                        for k, v in cached_svd.items(): new_columns[k] = v; generated_features.append(k)
                    else:
                        print(f"   -> [{esm_tag}_SVD50] Compressing 1280D to 50 dimensions using SVD...")
                        from sklearn.decomposition import TruncatedSVD
                        
                        # Reconstruct the 1280D matrix from cache, or use the one we just generated in memory
                        if cached_dict:
                            num_dims = len(cached_dict)
                            matrix = np.zeros((expected_dataset_len, num_dims))
                            for i in range(num_dims):
                                matrix[:, i] = cached_dict[f'{col}_{esm_tag}_{i}']
                        else:
                            matrix = embeddings
                            
                        # Safely set components (Max 50, but less if dataset is super small)
                        n_comps = min(50, matrix.shape[0] - 1, matrix.shape[1] - 1)
                        if n_comps > 0:
                            svd = TruncatedSVD(n_components=n_comps, random_state=42)
                            embeddings_svd = svd.fit_transform(matrix)
                            svd_dict = {f'{col}_{esm_tag}_SVD50_{i}': embeddings_svd[:, i] for i in range(embeddings_svd.shape[1])}
                            
                            if not is_inference: np.savez(svd_cache, **svd_dict)
                            for k, v in svd_dict.items(): new_columns[k] = v; generated_features.append(k)
                        else:
                            print(f"   -> ⚠️ Dataset too small for SVD compression. Skipping.")

        # ==========================================
        # 🌟 NEW: AntiBERTy (512D)
        # ==========================================
        if ANTIBERTY_AVAILABLE:
            antiberty_cache = os.path.join(cache_dir, f"{col}_AntiBERTy_features_N{expected_dataset_len}.npz")
            cached_dict = _load_valid_cache(antiberty_cache, expected_dataset_len) if not is_inference else None
            
            if cached_dict:
                print("   -> [AntiBERTy] Loaded from cache")
                for k, v in cached_dict.items(): new_columns[k] = v; generated_features.append(k)
            else:
                print("   -> [AntiBERTy] Generating antibody-specific embeddings...")
                import torch
                antiberty = AntiBERTyRunner()
                antiberty.model.eval()
                
                antiberty_embeddings = []
                for seq in seqs:
                    if seq == 'NAN' or len(seq) < 2:
                        antiberty_embeddings.append(np.zeros(512))
                    else:
                        with torch.no_grad():
                            emb = antiberty.embed([seq])[0]
                            # Mean pool, ignoring <CLS> and <SEP> tokens at the ends
                            if emb.shape[0] > 2: emb_mean = emb[1:-1].mean(dim=0).cpu().numpy()
                            else: emb_mean = emb.mean(dim=0).cpu().numpy()
                            antiberty_embeddings.append(emb_mean)
                            
                antiberty_embeddings = np.array(antiberty_embeddings)
                antiberty_dict = {f'{col}_AntiBERTy_{i}': antiberty_embeddings[:, i] for i in range(antiberty_embeddings.shape[1])}
                if not is_inference: np.savez(antiberty_cache, **antiberty_dict)
                for k, v in antiberty_dict.items(): new_columns[k] = v; generated_features.append(k)

        # Georgiev (19D)
        geo_cache = os.path.join(cache_dir, f"{col}_georgiev_features_N{expected_dataset_len}.npz")
        cached_dict = _load_valid_cache(geo_cache, expected_dataset_len)
        if cached_dict:
            print("   -> [Georgiev] Loaded from cache")
            for k, v in cached_dict.items():
                new_columns[k] = v
                if k not in generated_features: generated_features.append(k)
        else:
            print("   -> [Georgiev] Calculating new features...")
            geo_features = []
            for seq in seqs:
                if seq == 'NAN' or len(seq) == 0: geo_features.append([0.0] * 19)
                else:
                    seq_geo = [GEORGIEV_DICT[aa] for aa in seq if aa in GEORGIEV_DICT]
                    geo_features.append(np.mean(seq_geo, axis=0).tolist() if seq_geo else [0.0] * 19)
            
            geo_features = np.array(geo_features)
            new_geo_dict = {f"{col}_Georgiev_PC{i+1}": geo_features[:, i] for i in range(19)}
            np.savez(geo_cache, **new_geo_dict)
            for k, v in new_geo_dict.items():
                new_columns[k] = v
                if k not in generated_features: generated_features.append(k)

    # ==========================================
    # 🌟 NEW: AbLang2 PAIRED EXTRACTION (VH + VL only)
    # ==========================================
    if ABLANG2_AVAILABLE:
        print("\n⚙️  Processing region: Paired [Global_VH + Global_VL]")
        ablang2_cache = os.path.join(cache_dir, f"Paired_VH_VL_AbLang2_features_N{expected_dataset_len}.npz")
        cached_dict = _load_valid_cache(ablang2_cache, expected_dataset_len) if not is_inference else None
        
        if cached_dict:
            print("   -> [AbLang2 Paired] Loaded from cache")
            for k, v in cached_dict.items(): new_columns[k] = v; generated_features.append(k)
        else:
            print("   -> [AbLang2 Paired] Generating Heavy/Light joint embeddings...")
            try:
                # 🌟 FIX 1: Be perfectly explicit with the parameter name
                ablang = ablang2.pretrained(model_to_use="ablang2-paired", random_init=False)
                
                heavy_seqs = df_feat['Global_VH'].astype(str).str.replace(r'\s+|,', '', regex=True).str.upper().tolist()
                light_seqs = df_feat['Global_VL'].astype(str).str.replace(r'\s+|,', '', regex=True).str.upper().tolist()
                
                ablang_embeddings = []
                emb_dim = None
                
                for h_seq, l_seq in zip(heavy_seqs, light_seqs):
                    if h_seq in ['NAN', 'NONE', ''] or l_seq in ['NAN', 'NONE', ''] or len(h_seq) < 2 or len(l_seq) < 2:
                        ablang_embeddings.append(None) # Placeholder, fill later when we know the dimension
                    else:
                        # 🌟 FIX 2: Pass as a "List of Lists" so Python doesn't confuse the arguments!
                        res = ablang([[h_seq, l_seq]], mode='seqcoding')
                        
                        emb = res[0]
                        if hasattr(emb, 'cpu'): emb = emb.cpu()
                        if hasattr(emb, 'numpy'): emb = emb.numpy()
                        ablang_embeddings.append(emb)
                        if emb_dim is None:
                            emb_dim = emb.shape[0]
                
                # Fallback if ALL sequences failed (e.g., terrible dataset)
                if emb_dim is None: emb_dim = 480 
                
                # Replace None placeholders with proper zeros
                final_embeddings = [emb if emb is not None else np.zeros(emb_dim) for emb in ablang_embeddings]
                final_embeddings = np.array(final_embeddings)
                
                ablang_dict = {f'Paired_VH_VL_AbLang2_{i}': final_embeddings[:, i] for i in range(final_embeddings.shape[1])}
                if not is_inference: np.savez(ablang2_cache, **ablang_dict)
                for k, v in ablang_dict.items(): new_columns[k] = v; generated_features.append(k)
                
            except Exception as e:
                print(f"   -> ⚠️ [AbLang2 Paired] Failed to generate. Error: {e}")
    if new_columns:
        df_feat = pd.concat([df_feat, pd.DataFrame(new_columns)], axis=1)
        
    print("\n✅ FEATURE EXTRACTION COMPLETE!")
    return df_feat, seq_cols, generated_features, aaindex_desc

def get_model(model_name, n_features, transform_type=None, weight_col=None):
    scoring = {'rmse': 'neg_root_mean_squared_error', 'mae': 'neg_mean_absolute_error', 'r2': 'r2', 'spearman': make_scorer(custom_spearman)}
    
    if model_name == 'RandomForest':
        base = RandomForestRegressor(n_estimators=100, random_state=42)
        grid = {}
    elif model_name == 'PLSRegression':
        base = Pipeline([('vt', VarianceThreshold()), ('scaler', StandardScaler()), ('pls', PLSRegression())])
        max_comp = min(10, n_features)
        grid = {'pls__n_components': range(2, max_comp + 1)} if max_comp >= 2 else {}
        if max_comp < 2: base.set_params(pls__n_components=1)
    elif model_name == 'ElasticNet':
        base = Pipeline([('vt', VarianceThreshold()), ('scaler', StandardScaler()), ('enet', ElasticNet(max_iter=10000, random_state=42))])
        grid = {'enet__alpha': [0.01, 0.1, 1.0, 10.0], 'enet__l1_ratio': [0.1, 0.5, 0.9]}
    elif model_name == 'SVR':
        base = Pipeline([('vt', VarianceThreshold()), ('scaler', StandardScaler()), ('svr', SVR())])
        grid = {'svr__kernel': ['linear'], 'svr__C': [0.001, 0.01, 0.1, 1.0], 'svr__epsilon':[0.01, 0.1, 0.5, 1.0]}
    elif model_name == 'XGBoost':
        base = Pipeline(steps=[
            ('vt', VarianceThreshold()),
            ('scaler', StandardScaler()),
            # n_jobs=1 inside the model prevents clashes with GridSearchCV's n_jobs=-1
            ('model', XGBRegressor(random_state=42, n_jobs=1, objective='reg:squarederror'))
        ])
        
        # 🌟 STRICT REGULARIZATION GRID: Designed specifically to prevent scaffold memorization
        grid = {
            'model__n_estimators': [200],#[50, 100, 200],    
            'model__max_depth': [6],#[2, 3],               # VERY shallow trees (prevents complex memorization rules)
            'model__learning_rate': [0.3],#[0.01, 0.05, 0.1],
            'model__subsample': [1],#[0.6, 0.8],           # Forces the model to ignore 20-40% of the sequences per tree
            'model__colsample_bytree': [1],#[0.5, 0.8],    # Forces the model to ignore 20-50% of the ESM features per tree
            'model__reg_alpha': [0],#[0.1, 1.0],           # L1 (Lasso) Penalty to crush useless features to 0
            'model__reg_lambda': [1.0]#[1.0, 5.0, 10.0]     # L2 (Ridge) Penalty to keep feature weights small and stable
        }
        
    else:
        raise ValueError(f"Unsupported model_name: {model_name}")

    model = SelfContainedTargetTransformRegressor(regressor=base, transform_type=transform_type, weight_col=weight_col)
    grid = {f'regressor__{k}': v for k, v in grid.items()}
    
    return GridSearchCV(model, grid, cv=3, scoring=scoring, refit='spearman', n_jobs=1) if grid else model

def plot_enrichment_comparison(y_true, y_pred, target_col, ax=None, top_quantile=0.2):
    if ax is None: fig, ax = plt.subplots(figsize=(8, 6))
    
    # Flatten arrays to prevent multi-dimensional Pandas errors
    y_true_flat = np.asarray(y_true).flatten()
    y_pred_flat = np.asarray(y_pred).flatten()
    
    df = pd.DataFrame({'Actual': y_true_flat, 'Predicted': y_pred_flat})
    total_library = len(df)
    target_lower = target_col.lower()
    lower_is_better = any(t in target_lower for t in ['hmw', 'agg', 'viscosity', 'lmw', 'hcp', 'clearance'])
    
    df_sorted = df.sort_values(by='Predicted', ascending=lower_is_better).reset_index(drop=True)
    total_samples = len(df)
    
    true_top_threshold = df['Actual'].quantile(top_quantile) if lower_is_better else df['Actual'].quantile(1 - top_quantile)
    
    enrichment_counts = []
    x_abs_counts = list(range(1, total_samples + 1))
    
    for k in x_abs_counts:
        top_k = df_sorted.head(k)
        if lower_is_better: hits = (top_k['Actual'] <= true_top_threshold).sum()
        else: hits = (top_k['Actual'] >= true_top_threshold).sum()
        enrichment_counts.append(hits)
    total_top_performers = enrichment_counts[-1]    
    random_expected = [k * top_quantile for k in x_abs_counts]
    
    ax.plot(x_abs_counts, enrichment_counts, label=f'Model Selection Hits', color='blue', lw=2.5)
    ax.plot(x_abs_counts, random_expected, 'k--', label='Random Selection (Expected)', lw=2)
    
    # 🌟 NEW: Shade the area between the model and random expectation to highlight the Enrichment Gain!
    ax.fill_between(x_abs_counts, enrichment_counts, random_expected, color='dodgerblue', alpha=0.15)

    # 🌟 NEW: Set strict boundaries to frame the plot perfectly
    
    ax.set_xlim(0, total_library+1)
    ax.set_ylim(0, total_top_performers+1)
    
    # 🌟 NEW: Force the exact max integers on the X-axis (preventing text overlap)
    x_ticks = [int(t) for t in ax.get_xticks() if 0 <= t < total_library * 0.95]
    x_ticks.append(total_library)
    ax.set_xticks(x_ticks)
    
    # 🌟 NEW: Force the exact max integers on the Y-axis
    y_ticks = [int(t) for t in ax.get_yticks() if 0 <= t < total_top_performers * 0.95]
    y_ticks.append(total_top_performers)
    ax.set_yticks(y_ticks)
    
    ax.set_title(f'Enrichment of Top {int(top_quantile*100)}% Hits\n({target_col})', fontsize=14)
    ax.set_xlabel('Number of Antibodies Synthesized / Tested', fontsize=12)
    ax.set_ylabel('Absolute Number of True Hits Discovered', fontsize=12)
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)

def plot_best_model_diagnostics(X, y, subregions_name, features_name, model_name, target_col, output_dir, final_estimator, 
                                best_params=None, feature_tag="", prefix="", top_quantile=0.2, hue_data=None, hue_name=None):
    """Clean 1x2 Grid: Scatter (with inline metrics) + Enrichment Plot."""
    print(f"\nGenerating diagnostic plots for {subregions_name} + {features_name}...")
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    # 🌟 NEW: Beautiful, explicit two-line title breakdown!
    title_text = f"Diagnostic Analysis: {target_col}\nModel: {model_name}  |  Regions: [{subregions_name}]  | Features: [{features_name}]"
    fig.suptitle(title_text, fontsize=18, y=1.05, fontweight='bold')
    
    try:
        # Rigorous Leave-One-Out CV for the scatter plot
        loo = 3#LeaveOneOut()
        cv_preds = cross_val_predict(final_estimator, X, y, cv=loo, n_jobs=-1)
        
        # 🌟 NEW: Flatten arrays here too so scikit-learn metrics and pandas don't crash
        y_flat = np.asarray(y).flatten()
        cv_preds_flat = np.asarray(cv_preds).flatten()
        
        c_r2, c_rmse, c_mae, c_spear = r2_score(y_flat, cv_preds_flat), np.sqrt(mean_squared_error(y_flat, cv_preds_flat)), mean_absolute_error(y_flat, cv_preds_flat), custom_spearman(y_flat, cv_preds_flat)
        
        metrics_text = f"Spearman: {c_spear:.3f}\nR² Score: {c_r2:.3f}\nRMSE: {c_rmse:.2f}\nMAE: {c_mae:.2f}"
        if best_params: metrics_text += "\n\nHyperparameters:\n" + "\n".join([f"{k.split('__')[-1]}: {v}" for k, v in best_params.items()])
            
        plot_df = pd.DataFrame({'Actual': y_flat, 'Predicted': cv_preds_flat})
            
        if hue_data is not None and hue_name is not None:
            plot_df[hue_name] = np.asarray(hue_data).flatten()
            sns.scatterplot(data=plot_df, x='Actual', y='Predicted', hue=hue_name, palette='tab10', ax=axes[0], alpha=0.8, edgecolor='k', s=80)
        else:
            sns.scatterplot(data=plot_df, x='Actual', y='Predicted', ax=axes[0], alpha=0.8, edgecolor='k', s=80, color='dodgerblue')
        
        min_val, max_val = min(y_flat.min(), cv_preds_flat.min()), max(y_flat.max(), cv_preds_flat.max())
        axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.5, label="Perfect Prediction")
        
        props = dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.9, edgecolor='gray')
        axes[0].text(0.05, 0.95, metrics_text, transform=axes[0].transAxes, fontsize=11, verticalalignment='top', bbox=props)
        # axes[0].set_title('Predicted vs Actual (LOO CV)')
        axes[0].set_title('Predicted vs Actual (3-Fold CV)')

        axes[0].legend(loc='lower right')

        plot_enrichment_comparison(y_flat, cv_preds_flat, target_col, ax=axes[1], top_quantile=top_quantile)
        
    except Exception as e: print(f"Warning: Plot failed: {e}")
        
    plt.tight_layout()
    feat_suffix = f"_{feature_tag}" if feature_tag else ""
    plt.savefig(os.path.join(output_dir, f'best_{prefix}{model_name}_{target_col}{feat_suffix}_diagnostics.png'), dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

def plot_best_model_diagnostics_old(X, y, subregions_name, features_name, model_name, target_col, output_dir, final_estimator, best_params=None, threshold=None, transform_type=None, feature_tag="", hue_data=None, hue_name=None, prefix="", top_quantile=0.2):
    """Generates the massive 3x3 diagnostic and learning curve plots for the best model."""
    transform_suffix = f"_{transform_type}" if transform_type else ""
    feat_suffix = f"_{feature_tag}" if feature_tag else ""
    title_tag = f" ({transform_type.title()} Transformed)" if transform_type else ""
    
    combo_name = f"{subregions_name} | {features_name}"
    print(f"\nGenerating EXTENDED diagnostic plots for ({combo_name}) using {model_name} on {target_col}{title_tag}...")
    
    # 🌟 EXPANDED: 3x3 Grid (9 slots total)
    fig, axes = plt.subplots(2, 3, figsize=(24, 12))
    fig.suptitle(f'Diagnostic Plots: Best Model ({combo_name} | {model_name} | {target_col}){title_tag}', fontsize=18, y=0.98)

    try:
        loo = LeaveOneOut()
        cv_preds = cross_val_predict(final_estimator, X, y, cv=loo, n_jobs=-1)

        # 🌟 NEW: Safely flatten arrays before scikit-learn metrics
        y_flat = np.asarray(y).flatten()
        cv_preds_flat = np.asarray(cv_preds).flatten()
        residuals = y_flat - cv_preds_flat
        calc_r2 = r2_score(y_flat, cv_preds_flat)
        calc_rmse = np.sqrt(mean_squared_error(y_flat, cv_preds_flat))
        calc_mae = mean_absolute_error(y_flat, cv_preds_flat)
        calc_spearman = custom_spearman(y_flat, cv_preds_flat)
        if threshold is None:
            threshold = np.median(y_flat)
            
        y_binary = (y_flat >= threshold).astype(int)
        cv_preds_binary = (cv_preds_flat >= threshold).astype(int)
        acc = accuracy_score(y_binary, cv_preds_binary)
        prec = precision_score(y_binary, cv_preds_binary, zero_division=0)
        rec = recall_score(y_binary, cv_preds_binary, zero_division=0)
        f1 = f1_score(y_binary, cv_preds_binary, zero_division=0)
        cm = confusion_matrix(y_binary, cv_preds_binary)

        try:
            roc_auc = roc_auc_score(y_binary, cv_preds_flat)
            pr_auc = average_precision_score(y_binary, cv_preds_flat)
            auc_text = f"ROC-AUC: {roc_auc:.2f} | PR-AUC: {pr_auc:.2f}\n"
        except ValueError:
            auc_text = "ROC/PR-AUC: N/A (Single Class)\n"
            
        metrics_text = (
            f"Spearman: {calc_spearman:.3f}\n"
            f"R²: {calc_r2:.3f}\n"
            f"RMSE: {calc_rmse:.2f}\n"
            f"MAE: {calc_mae:.2f}"
        )

        metrics_text += f"\n\nBinary Eval (Cutoff: {threshold:.2f}):\n{auc_text}Acc: {acc:.2f} | F1: {f1:.2f} | P: {prec:.2f} | R: {rec:.2f}"
        if best_params:
            params_str = "\n".join([f"{k.split('__')[-1]}: {v}" for k, v in best_params.items()])
            metrics_text += f"\n\nOptimal Params:\n{params_str}"
            
        # ==========================================
        # ROW 1: REGRESSION ACCURACY & ERROR
        # ==========================================

        classification_labels = []
        for actual, pred in zip(y_flat, cv_preds_flat):
            if actual >= threshold and pred >= threshold:
                classification_labels.append('True Positive (TP)')
            elif actual < threshold and pred < threshold:
                classification_labels.append('True Negative (TN)')
            elif actual < threshold and pred >= threshold:
                classification_labels.append('False Positive (FP)')
            else:
                classification_labels.append('False Negative (FN)')
        plot_df = pd.DataFrame({'Actual': y_flat, 'Predicted': cv_preds_flat, 'Classification': classification_labels})
        plot_df_res = pd.DataFrame({'Predicted': cv_preds_flat, 'Residuals': residuals})

        style_col = None

        if hue_data is not None and hue_name is not None:
            style_col = hue_name
            plot_df[style_col] = np.asarray(hue_data).flatten()
            plot_df_res[style_col] = np.asarray(hue_data).flatten()

        custom_palette = {
            'True Negative (TN)': '#2ca02c', 'True Positive (TP)': '#d62728',
            'False Positive (FP)': '#ff7f0e', 'False Negative (FN)': '#1f77b4'
        }

        # --- Plot 1: Predicted vs Actual ---
        sns.scatterplot(data=plot_df, x='Actual', y='Predicted', hue='Classification',
                        style=style_col, palette=custom_palette, ax=axes[0, 0], alpha=0.8, edgecolor='k', s=60)
        axes[0, 0].axvline(threshold, color='gray', linestyle=':', linewidth=1.5, label=f'Threshold ({threshold:.1f})')
        axes[0, 0].axhline(threshold, color='gray', linestyle=':', linewidth=1.5)
        min_val = min(y_flat.min(), cv_preds_flat.min())
        max_val = max(y_flat.max(), cv_preds_flat.max())
        axes[0, 0].plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.5)
        axes[0, 0].set_title(f'Predicted vs Actual {target_col} (LOO CV)')
        axes[0, 0].set_xlabel(f'Actual {target_col}')
        axes[0, 0].set_ylabel(f'Predicted {target_col} (Out-of-Fold)')
        axes[0, 0].legend(loc='lower right', fontsize=9)

        # --- Plot 2: Residuals vs Predicted ---
        sns.scatterplot(data=plot_df_res, x='Predicted', y='Residuals',
                        style=style_col, color='purple', ax=axes[0, 1], alpha=0.7, edgecolor='k', s=60)
        axes[0, 1].axhline(0, color='r', linestyle='--')
        axes[0, 1].set_title('Residuals vs Predicted')
        axes[0, 1].set_xlabel(f'Predicted {target_col}')
        axes[0, 1].set_ylabel('Residuals (Actual - Predicted)')
        if style_col:
            axes[0, 1].legend(loc='lower right', fontsize=9)
        # --- Plot 3: Distribution of Residuals ---
        sns.histplot(residuals, kde=True, ax=axes[0, 2], color='purple', bins=15)
        axes[0, 2].axvline(0, color='r', linestyle='--')
        axes[0, 2].set_title('Distribution of Residuals')
        axes[0, 2].set_xlabel(f'Residual Error {target_col}')
        axes[0, 2].set_ylabel('Frequency')
        # ==========================================
        # ROW 2: TRANSLATIONAL VALUE & RANKING
        # ==========================================
        # --- Plot 4: Confusion Matrix ---
        dummy_matrix = np.array([[0, 1], [2, 3]])
        cm_cmap = ListedColormap(['#2ca02c', '#ff7f0e', '#1f77b4', '#d62728'])
        sns.heatmap(dummy_matrix, annot=cm, fmt='d', cmap=cm_cmap, ax=axes[1, 0], cbar=False,
                    xticklabels=[f'<{threshold:.1f}', f'>={threshold:.1f}'],
                    yticklabels=[f'<{threshold:.1f}', f'>={threshold:.1f}'],
                    annot_kws={"size": 22, "weight": "bold", "color": "white"})
        axes[1, 0].set_title(f'Classification Confusion Matrix\n(Threshold = {threshold:.2f})')
        axes[1, 0].set_xlabel('Predicted Class')
        axes[1, 0].set_ylabel('Actual Class')
        # --- Plot 5: Enrichment Comparison ---
        plot_enrichment_comparison(y_flat, cv_preds_flat, target_col, ax=axes[1, 1], top_quantile=top_quantile)
        # ==========================================
        # ROW 3: SUMMARY & METRICS DASHBOARD
        # ==========================================
        axes[1, 2].axis('off')  
        props = dict(boxstyle='round,pad=1', facecolor='#f8f9fa', alpha=1.0, edgecolor='gray', linewidth=2)
        axes[1, 2].text(0.5, 0.5, metrics_text, transform=axes[1, 2].transAxes,
                        fontsize=14, verticalalignment='center', horizontalalignment='center', bbox=props)
        axes[1, 2].set_title("Model Performance Summary", fontsize=16, pad=20)
    except Exception as e:
        print(f"Warning: Could not generate LOO predictions due to mathematical failure: {e}")
        for row in range(2):
            for col in range(3):
                axes[row, col].set_title("Plot Failed")
                axes[row, col].text(0.5, 0.5, f"Error: {e}", ha='center', va='center', wrap=True)
    plt.tight_layout(rect=[0, 0.03, 1, 0.96])
    plot_filename = os.path.join(output_dir, f'best_{prefix}{model_name}_{target_col}{transform_suffix}{feat_suffix}_EXTENDED_diagnostics.png')

    plt.savefig(plot_filename, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved Extended Plot '{plot_filename}'")
    
    # print("Generating comprehensive learning curves (this takes a moment)...")
    # fig_lc, axes_lc = plt.subplots(2, 2, figsize=(16, 12))
    # fig_lc.suptitle(f'Learning Curves: Best Model ({combo_name} | {model_name} | {target_col}){title_tag}', fontsize=16)
    # metrics_to_plot = {
    #     'Negative RMSE': 'neg_root_mean_squared_error',
    #     'Negative MAE': 'neg_mean_absolute_error',
    #     'R2 Score': 'r2',
    #     'Spearman Correlation': make_scorer(custom_spearman)
    # }
    # try:
    #     for ax, (name, scorer) in zip(axes_lc.flatten(), metrics_to_plot.items()):
    #         LearningCurveDisplay.from_estimator(
    #             final_estimator, X, y, cv=5, n_jobs=-1,
    #             train_sizes=np.linspace(0.2, 1.0, 10),
    #             scoring=scorer,
    #             error_score=np.nan,
    #             ax=ax
    #         )
    #         ax.set_title(f'Learning Curve ({name})')
    #         ax.legend(loc='best')
    # except Exception as e:
    #     print(f"    -> Warning: Learning curves failed due to math error: {e}")
    #     for ax in axes_lc.flatten():
    #         ax.set_title("Learning Curve Failed")
    # plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    # lc_filename = os.path.join(output_dir, f'best_{prefix}{model_name}_{target_col}{transform_suffix}{feat_suffix}_learning_curves.png')
    # plt.savefig(lc_filename, dpi=300, facecolor='white')
    # plt.close()
    # print(f"Saved Learning Curves '{lc_filename}'") 

def plot_out_of_group_diagnostics(X, y, group_labels, subregions_name, features_name, model_name, target_col, output_dir, final_estimator, feature_tag="", prefix="", split_col_name=""):
    """Trains on Group A -> Tests on all other Groups. Plots the out-of-group generalization in consolidated subplots."""
    
    print(f"\nGenerating Out-of-Group (OOG) diagnostic plots for {target_col}...")
    
    group_labels = np.asarray(group_labels).flatten()
    y_flat = np.asarray(y).flatten()
    
    # --- NEW: Filter out ignored/empty groups ---
    # Catch explicit ignore tags, as well as Pandas NaNs and empty strings
    ignore_values = ['IGNORE', 'EXCLUDE']#, 'NA', 'NAN', 'NONE', '']
    
    valid_mask = []
    for val in group_labels:
        if pd.isna(val):
            valid_mask.append(False)
        elif str(val).strip().upper() in ignore_values:
            valid_mask.append(False)
        else:
            valid_mask.append(True)
            
    valid_mask = np.array(valid_mask)
    
    # Apply the filter to X, y, and the group labels before doing any math
    group_labels = group_labels[valid_mask]
    y_flat = y_flat[valid_mask]
    
    if hasattr(X, 'iloc'):
        X = X.iloc[valid_mask]
    else:
        X = X[valid_mask]
        
    unique_groups = np.unique(group_labels)
    num_groups = len(unique_groups)
    
    print(f"🔍 Detected {num_groups} valid unique groups in split column '{split_col_name}': {list(unique_groups)}")
    
    if num_groups < 2:
        print("  -> ⚠️ Not enough groups to perform out-of-group testing.")
        return
        
    if num_groups > 10:
        print(f"  -> ⚠️ Skipping out-of-group combinations: {num_groups} groups is too many (Limit is 10 to prevent plot explosion).")
        return

    # Determine Grid Size for Subplots (Max 3 columns wide)
    cols = min(3, num_groups)
    rows = math.ceil(num_groups / cols)
    
    fig_scatter, axes = plt.subplots(rows, cols, figsize=(7 * cols, 6 * rows))
    # Ensure axes is always a flat array for easy iteration, even if it's 1x1 or 1xN
    axes = np.array(axes).flatten()
    
    title_text = f"Generalization: {target_col}\nModel: {model_name}  |  Regions: [{subregions_name}]"
    fig_scatter.suptitle(title_text, fontsize=18, y=1.02 + (0.02 if rows==1 else 0), fontweight='bold')
    
    results = []
    feat_suffix = f"_{feature_tag}" if feature_tag else ""
    split_name_tag = f"split_by_{split_col_name}_" if split_col_name else ""
    
    for idx, train_grp in enumerate(unique_groups):
        ax = axes[idx]
        mask_train = (group_labels == train_grp)
        
        # Safely index X
        if hasattr(X, 'iloc'):
            X_train = X.iloc[mask_train]
        else:
            X_train = X[mask_train]
            
        y_train = y_flat[mask_train]
        
        if len(y_train) < 5:
            print(f"     ⚠️ Skipping Train [{train_grp}] (Insufficient data: N={len(y_train)})")
            ax.set_visible(False)
            continue
            
        # Train fresh model on the current training group
        model = clone(final_estimator)
        model.fit(X_train, y_train)
        train_preds = model.predict(X_train).flatten()
        
        # Prepare data for this specific subplot
        plot_df_list = []
        plot_df_list.append(pd.DataFrame({
            'Actual': y_train, 
            'Predicted': train_preds, 
            'Dataset': f'Train ({train_grp})'
        }))
        
        metrics_lines = ["Unseen Test Metrics:"]
        palette = {f'Train ({train_grp})': 'lightgray'}
        
        # Generate a distinct color palette for the test groups
        test_colors = sns.color_palette("husl", num_groups - 1)
        color_idx = 0
        
        # Test on every OTHER group
        for test_grp in unique_groups:
            if test_grp == train_grp:
                continue
                
            mask_test = (group_labels == test_grp)
            if hasattr(X, 'iloc'):
                X_test = X.iloc[mask_test]
            else:
                X_test = X[mask_test]
                
            y_test = y_flat[mask_test]
            
            if len(y_test) < 5:
                continue
                
            test_preds = model.predict(X_test).flatten()
            
            # Calculate strictly on unseen test data
            c_r2 = r2_score(y_test, test_preds)
            try:
                c_spear = custom_spearman(y_test, test_preds)
            except Exception:
                c_spear = 0.0
                
            results.append({
                'Train_Group': train_grp,
                'Test_Group': test_grp,
                'Spearman': c_spear,
                'R2': c_r2,
                'N_Train': len(y_train),
                'N_Test': len(y_test)
            })
            
            print(f"     ✅ Train [{str(train_grp):^10}] ➔ Test [{str(test_grp):^10}] | Spearman: {c_spear:^6.3f} | R²: {c_r2:^6.3f}")
            
            dataset_label = f'Test ({test_grp})'
            plot_df_list.append(pd.DataFrame({
                'Actual': y_test, 
                'Predicted': test_preds, 
                'Dataset': dataset_label
            }))
            
            palette[dataset_label] = test_colors[color_idx]
            color_idx += 1
            metrics_lines.append(f"[{test_grp}] Sp: {c_spear:.2f} | R²: {c_r2:.2f}")

        # Plot the combined scatter
        plot_df = pd.concat(plot_df_list, ignore_index=True)
        sns.scatterplot(data=plot_df, x='Actual', y='Predicted', hue='Dataset', 
                        palette=palette, ax=ax, alpha=0.8, edgecolor='k', s=70)
        
        # Perfect prediction diagonal line
        min_val = plot_df[['Actual', 'Predicted']].min().min()
        max_val = plot_df[['Actual', 'Predicted']].max().max()
        ax.plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.5, label="Perfect Line")
        
        # Add metrics text box
        props = dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.9, edgecolor='gray')
        ax.text(0.05, 0.95, "\n".join(metrics_lines), transform=ax.transAxes, 
                fontsize=9.5, verticalalignment='top', bbox=props)
        
        ax.set_title(f'Trained strictly on Group: {train_grp}', fontsize=14, pad=10)
        ax.set_xlabel(f'Actual {target_col}')
        ax.set_ylabel(f'Predicted {target_col}')
        ax.legend(loc='lower right', fontsize=9)
        
    # Hide any unused subplots (if num_groups doesn't perfectly fill the grid)
    for idx in range(num_groups, len(axes)):
        axes[idx].set_visible(False)
        
    fig_scatter.tight_layout()
    scatter_file = os.path.join(output_dir, f"oog_{split_name_tag}{prefix}{model_name}_{target_col}{feat_suffix}_Combined_Scatter.png")
    fig_scatter.savefig(scatter_file, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close(fig_scatter)

    if len(results) > 0:
        res_df = pd.DataFrame(results)
        pivot_spearman = res_df.pivot(index='Train_Group', columns='Test_Group', values='Spearman')
        
        # Dynamic figure sizing for heatmap
        fig_width = max(8, num_groups * 1.5)
        fig_height = max(6, num_groups * 1.2)
        
        plt.figure(figsize=(fig_width, fig_height))
        
        sns.heatmap(pivot_spearman, annot=True, cmap='coolwarm', center=0, fmt=".2f", 
                    linewidths=1, linecolor='white',
                    cbar_kws={'label': 'Spearman Correlation Score'})
                    
        plt.title(f"OOG Generalization Heatmap (Spearman Rank)\nTarget: {target_col} | Model: {model_name}", fontsize=14, pad=20)
        plt.ylabel(f"Model Trained On ({split_col_name})", fontsize=12, fontweight='bold')
        plt.xlabel(f"Model Tested On ({split_col_name})", fontsize=12, fontweight='bold')
        
        heatmap_file = os.path.join(output_dir, f"oog_{split_name_tag}{prefix}{model_name}_{target_col}{feat_suffix}_Summary_Heatmap.png")
        plt.savefig(heatmap_file, dpi=300, bbox_inches='tight', facecolor='white')
        plt.close()
        
        print(f"\n  -> 📊 Saved 1 Combined OOG Scatter Plot and 1 Summary Heatmap to '{output_dir}/'")

def generate_shap_analysis(model, X, y, output_dir, feature_names, model_name, target_col, prefix="", feature_tag="", aaindex_desc=None):
    print(f"\nExtracting SHAP feature importances for {model_name}...")
    try:
        import shap
        import re
        
        # --- NEW: Sanitize Feature Names for XGBoost & SHAP ---
        # Strip accented characters (like 'É') and XGBoost-forbidden brackets
        clean_features = []
        for f in feature_names:
            f_clean = str(f).encode('ascii', 'ignore').decode('ascii')
            f_clean = re.sub(r'[\[\]<>]', '_', f_clean)
            clean_features.append(f_clean)
        feature_names = clean_features
        
        # 🌟 CRITICAL CRASH FIX: Actually rename the DataFrame columns!
        if isinstance(X, pd.DataFrame):
            X.columns = feature_names
        # ------------------------------------------------------
        
        # 🌟 CRITICAL SPEED FIX: Extract the winner BEFORE fitting!
        # This prevents running a massive GridSearch all over again.
        if hasattr(model, 'best_estimator_'):
            model = model.best_estimator_
            
        # Fit the single winning pipeline on 100% of data (Takes 1-2 seconds)
        model.fit(X, y)
        
        # Safely extract the base pipeline
        base_pipe = model.regressor_ if hasattr(model, 'regressor_') else model
        
        if hasattr(base_pipe, 'named_steps') and 'vt' in base_pipe.named_steps:
            X_trans = base_pipe.named_steps['vt'].transform(X)
            if 'scaler' in base_pipe.named_steps: X_trans = base_pipe.named_steps['scaler'].transform(X_trans)
            active_feats = np.array(feature_names)[base_pipe.named_steps['vt'].get_support()]
            predictor = base_pipe.named_steps[list(base_pipe.named_steps.keys())[-1]]
        else:
            X_trans, active_feats, predictor = X.values if isinstance(X, pd.DataFrame) else X, feature_names, base_pipe

        # --- Robust Tree Model Check ---
        is_tree = model_name in ['RandomForest', 'XGBoost'] or type(predictor).__name__ in ['RandomForestRegressor', 'XGBRegressor']

        if is_tree:
            print("  -> Using lightning-fast TreeExplainer...")
            explainer = shap.TreeExplainer(predictor)
            shap_values = explainer.shap_values(X_trans)[1] if isinstance(explainer.shap_values(X_trans), list) else explainer.shap_values(X_trans)
        elif model_name in ['PLSRegression', 'ElasticNet'] or (model_name == 'SVR' and getattr(predictor, 'kernel', '') == 'linear'):
            print("  -> Using exact linear math explainer...")
            coef = predictor.coef_.flatten()
            bg_mean = X_trans.mean(axis=0)
            shap_values = (X_trans - bg_mean) * coef
        else:
            print("  -> Non-linear model detected. Compressing background to 5 centroids to prevent RAM crash...")
            background = shap.kmeans(X_trans, 5)
            explainer = shap.KernelExplainer(predictor.predict, background)
            shap_values = explainer.shap_values(X_trans, nsamples=100, silent=True)

        # 🌟 NEW: Clean Y-axis + Side Dictionary Text Box 
        aac_desc = {
            'A': 'Alanine', 'R': 'Arginine', 'N': 'Asparagine', 'D': 'Aspartic Acid',
            'C': 'Cysteine', 'Q': 'Glutamine', 'E': 'Glutamic Acid', 'G': 'Glycine',
            'H': 'Histidine', 'I': 'Isoleucine', 'L': 'Leucine', 'K': 'Lysine',
            'M': 'Methionine', 'F': 'Phenylalanine', 'P': 'Proline', 'S': 'Serine',
            'T': 'Threonine', 'W': 'Tryptophan', 'Y': 'Tyrosine', 'V': 'Valine'
        }
        
        display_features = []
        legend_dict = {} 
        for f in active_feats:
            if '_AAindex_' in f and aaindex_desc:
                try:
                    base_col, code = f.split('_AAindex_')
                    clean_name = f"{code}"
                    display_features.append(clean_name)
                    # ALSO sanitize the dictionary description to be 100% safe
                    raw_desc = aaindex_desc.get(code, "Unknown property").split('(')[0].strip()
                    legend_dict[code] = raw_desc.encode('ascii', 'ignore').decode('ascii')
                except ValueError:
                    display_features.append(f)
            elif '_AAC_' in f:
                try:
                    base_col, aa = f.split('_AAC_')
                    clean_name = f"{base_col} | AAC_{aa}"
                    display_features.append(clean_name)
                    legend_dict[f"AAC_{aa}"] = f"{aac_desc.get(aa, 'Unknown')} Frequency"
                except ValueError:
                    display_features.append(f)
            else:
                display_features.append(f)

        # Calculate mean absolute SHAP values to find top 20 features
        mean_abs_shap = np.abs(shap_values).mean(axis=0)
        top_indices = np.argsort(mean_abs_shap)[-20:]
        top_features = [display_features[i] for i in top_indices]
        
        # Build the legend text strictly for the visible features
        legend_lines = ["Feature Dictionary:"]
        import textwrap
        for feat in reversed(top_features): # Top-to-bottom plot order
            # 1. Exact match (for AAindex codes like 'KRIW790101')
            if feat in legend_dict and feat not in str(legend_lines):
                wrapped_desc = textwrap.fill(legend_dict[feat], width=65)
                legend_lines.append(f"• {feat}: {wrapped_desc}")
            # 2. Substring match (for AAC like 'seq_cdrh3 | AAC_W')
            else:
                for key in legend_dict:
                    if key in feat and key not in str(legend_lines):
                        wrapped_desc = textwrap.fill(legend_dict[key], width=65)
                        legend_lines.append(f"• {key}: {wrapped_desc}")
                        break
                        
        legend_text = "\n".join(legend_lines)
        # 🌟 Make the figure wider (22 inches) to accommodate the horizontal text
        fig = plt.figure(figsize=(22, 8))
        # 🌟 Give the text box slightly more of the horizontal layout ratio
        gs = fig.add_gridspec(1, 2, width_ratios=[2, 1.2])
        ax_shap = fig.add_subplot(gs[0])
        ax_text = fig.add_subplot(gs[1])
        # Draw the SHAP plot on the left axis
        plt.sca(ax_shap) 
        shap.summary_plot(shap_values, X_trans, feature_names=display_features, show=False)
        ax_shap.set_title(f"SHAP Value Impact: {model_name} on {target_col}", fontsize=14)
        # Draw the Dictionary Text Box on the right axis
        ax_text.axis('off')
        if len(legend_lines) > 1:
            props = dict(boxstyle='round,pad=0.5', facecolor='#f8f9fa', alpha=0.9, edgecolor='gray')
            ax_text.text(0.0, 0.95, legend_text, transform=ax_text.transAxes, fontsize=10,
                         verticalalignment='top', bbox=props, family='monospace') 
        feat_suffix = f"_{feature_tag}" if feature_tag else ""
        plt.savefig(os.path.join(output_dir, f'shap_{prefix}{model_name}_{target_col}{feat_suffix}.png'), dpi=300, facecolor='white', bbox_inches='tight')
        plt.close()
        print(f"✅ SHAP Summary Plot successfully saved!")
    except MemoryError: print("⚠️ SHAP Memory Error: The KernelExplainer ran out of RAM. Skipping plot.")
    except Exception as e: print(f"⚠️ SHAP failed: {e}")

def filter_active_features(all_features, sub_combo, feat_combo, global_features):
    active = list(global_features)
    for f in all_features:
        if f in active: continue
        # 🌟 NEW: Safely grant CQA and Propermab VIP access, bypassing subregion checks
        if f.startswith('CQA_'):
            if 'CQA' in feat_combo: active.append(f)
            continue
        if f.startswith('Propermab_'):
            if 'Propermab' in feat_combo: active.append(f)
            continue

        # 🌟 NEW: Paired AbLang2 Bypass
        if f.startswith('Paired_VH_VL_AbLang2_'):
            # Only activate if the user explicitly requested AbLang2 AND they provided both VH and VL!
            if 'AbLang2_Paired' in feat_combo and 'Global_VH' in sub_combo and 'Global_VL' in sub_combo:
                active.append(f)
            continue
        
        # Must belong to one of the active subregions
        has_sub = any(f.startswith(sub + '_') for sub in sub_combo)
        if not has_sub: continue
            
        # Must belong to one of the active feature types
        markers = {'AAC': '_AAC_', 'AAindex': '_AAindex_', 'Georgiev': '_Georgiev_'}
        has_feat = False
        for ft in feat_combo:
            if ft in markers and markers[ft] in f: has_feat = True; break
            elif ft.startswith('ESM_'):
                # 🌟 NEW: Stop the SVD compressed features from accidentally leaking into the raw 650M test!
                if ft == 'ESM_Big_650M':
                    if '_ESM_Big_650M_' in f and '_SVD50_' not in f: has_feat = True; break
                elif f"_{ft}_" in f: has_feat = True; break
            elif ft == 'AntiBERTy' and '_AntiBERTy_' in f: has_feat = True; break
            # Note: AbLang2 is now handled by the bypass above, so we don't need it here!
            elif ft == 'Propermab' and f.startswith('Propermab_'): has_feat = True; break
            elif ft == 'CQA' and f.startswith('CQA_'): has_feat = True; break
                
        # Always include base features like Length that lack a marker
        is_untyped = not any(m in f for m in ['_AAC_', '_AAindex_', '_ESM_', '_Georgiev_', 'Propermab_', 'CQA_', '_AntiBERTy_', '_AbLang2_'])
        if has_feat or is_untyped: active.append(f)
            
    # 🌟 CRITICAL FIX: Alphabetically sort the features to completely defeat Python's set randomization!
    return sorted(list(set(active)))

def evaluate_exhaustive_combinations(df, seq_cols, generated_features, target_col, model_name, output_dir, 
                                     transform_type=None, weight_col=None, prefix="", hue_col=None, rank_to_plot=0,
                                     oog_col=None, aaindex_desc=None, extended_plots=False, manual_threshold=None):
    os.makedirs(output_dir, exist_ok=True)
    checkpoint_csv = os.path.join(output_dir, f"{prefix}exhaustive_search_checkpoint_{model_name}_{target_col}.csv")
    final_excel = os.path.join(output_dir, f"{prefix}exhaustive_search_results_{model_name}_{target_col}.xlsx")
    
    global_features = [
        f for f in generated_features 
        if not f.startswith('seq_')
        and not f.startswith('Global_')
        and not f.startswith('CQA_')
        and not f.startswith('Propermab_')
        and not f.startswith('Paired_') 
    ]
    
    available_groups = []
    if any('_AAC_' in f for f in generated_features): available_groups.append('AAC')
    if any('_AAindex_' in f for f in generated_features): available_groups.append('AAindex')
    if any('_ESM_Small_8M_' in f for f in generated_features): available_groups.append('ESM_Small_8M')
    if any('_ESM_Medium_35M_' in f for f in generated_features): available_groups.append('ESM_Medium_35M')
    
    # 🌟 NEW: Fixed python syntax bug that would have thrown a NameError here!
    if any(('_ESM_Big_650M_' in f and '_SVD50_' not in f) for f in generated_features): available_groups.append('ESM_Big_650M')
    if any('_ESM_Big_650M_SVD50_' in f for f in generated_features): available_groups.append('ESM_Big_650M_SVD50')
    if any('_AntiBERTy_' in f for f in generated_features): available_groups.append('AntiBERTy')
    # 🌟 NEW: Check for Paired AbLang2 features
    if any('Paired_VH_VL_AbLang2_' in f for f in generated_features): available_groups.append('AbLang2_Paired')
    if any('_Georgiev_' in f for f in generated_features): available_groups.append('Georgiev')
    if any(f.startswith('Propermab_') for f in generated_features): available_groups.append('Propermab')
    if any(f.startswith('CQA_') for f in generated_features): available_groups.append('CQA')
    
    completed_combos = set()
    file_exists = os.path.isfile(checkpoint_csv)
    if file_exists:
        try:
            df_check = pd.read_csv(checkpoint_csv)
            for _, row in df_check.iterrows(): completed_combos.add(f"{row['Subregions']}|{row['Features']}")
            print(f"✅ Found Checkpoint! Resuming search. {len(completed_combos)} combinations already completed.")
        except Exception: file_exists = False

    with open(checkpoint_csv, 'a', newline='') as f:
        writer = csv.writer(f)
        if not file_exists: writer.writerow(['Subregions', 'Features', 'Num_Features', 'Spearman', 'Spearman_Std', 'RMSE', 'RMSE_Std', 'MAE', 'MAE_Std', 'R2', 'R2_Std'])
            
        scorer = {'spearman': make_scorer(custom_spearman), 'r2': 'r2', 'rmse': 'neg_root_mean_squared_error', 'mae': 'neg_mean_absolute_error'}
        cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)
        
        all_cols = df.columns.tolist()
        
        # --- NEW: Pre-calculate valid feature combinations so the progress bar is 100% accurate! ---
        valid_feat_combos = []
        for FL in range(1, len(available_groups) + 1):
            for feat_combo in itertools.combinations(available_groups, FL):
                # 🌟 NEW: Only allow AT MOST ONE heavy language model per combination to prevent overfitting 
                lm_count = sum(1 for g in feat_combo if g.startswith('ESM_') or g in ['AntiBERTy', 'AbLang2_Paired'])
                if lm_count <= 1:
                    valid_feat_combos.append(feat_combo)
                    
        # --- NEW: Pre-calculate valid subregion combinations for an exact progress count! ---
        valid_sub_combos = []
        for L in range(1, len(seq_cols) + 1):
            for sub_combo in itertools.combinations(seq_cols, L):
                # Failsafe! Never combine the whole Fv with its individual parts.
                if 'Global_Fv' in sub_combo and len(sub_combo) > 1:
                    continue
                valid_sub_combos.append(sub_combo)

        # 🌟 NEW: Calculate the exact valid experiments by checking the AbLang2 guardrails!
        valid_experiments = []
        for sub_combo in valid_sub_combos:
            for feat_combo in valid_feat_combos:
                
                # The AbLang2 Strict Guardrail
                if 'AbLang2_Paired' in feat_combo:
                    # It MUST contain both Global_VH and Global_VL
                    if 'Global_VH' not in sub_combo or 'Global_VL' not in sub_combo:
                        continue
                    # It MUST NOT contain the stitched Fv string (which ruins the paired logic)
                    if 'Global_Fv' in sub_combo:
                        continue
                        
                valid_experiments.append((sub_combo, feat_combo))
            
        # 🌟 CRITICAL FIX: Calculate total runs based on the filtered list, not the raw multiplication!
        total_runs = len(valid_experiments)
        current_run = 0

        print(f"\n🚀 Starting Exhaustive Search: {total_runs} Total Valid Combinations...")
        for sub_combo, feat_combo in valid_experiments:
            sub_name = " + ".join(sub_combo)
            feat_name = " + ".join(feat_combo)
            combo_id = f"{sub_name}|{feat_name}"
            current_run += 1

            if combo_id in completed_combos: continue

            selected_cols = filter_active_features(all_cols, sub_combo, feat_combo, global_features)
            # 🌟 NEW: Always inject the Single Media feature into the selected columns
            if 'Media_Encoded' in all_cols and 'Media_Encoded' not in selected_cols:
                selected_cols.append('Media_Encoded')
                if 'Media' not in feat_combo:
                    feat_combo = list(feat_combo) + ['Media']

            cols_to_check = [target_col] + selected_cols
            if weight_col and weight_col in df.columns: cols_to_check.append(weight_col)
            model_df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=cols_to_check).reset_index(drop=True)
            
            if len(model_df) < 15 or len(selected_cols) == len(global_features):
                writer.writerow([sub_name, feat_name, len(selected_cols), 0, 0, 0, 0, 0, 0, 0, 0])
                f.flush()
                continue

            X = model_df[selected_cols + ([weight_col] if weight_col else [])]
            y = model_df[target_col]
            
            model = get_model(model_name, len(selected_cols), transform_type, weight_col)
            print(f"[{current_run}/{total_runs}] Testing: [{sub_name}] with [{feat_name}] ({len(selected_cols)} feats)...")
            
            scores = cross_validate(model, X, y, cv=cv, scoring=scorer, n_jobs=-1, error_score=np.nan)
            valid = ~np.isnan(scores['test_r2'])
            
            if valid.sum() == 0: 
                writer.writerow([sub_name, feat_name, len(selected_cols), 0, 0, 0, 0, 0, 0, 0, 0])
            else:
                writer.writerow([
                    sub_name, feat_name, len(selected_cols), 
                    np.mean(scores['test_spearman'][valid]), np.std(scores['test_spearman'][valid]),
                    -np.mean(scores['test_rmse'][valid]), np.std(scores['test_rmse'][valid]), 
                    -np.mean(scores['test_mae'][valid]), np.std(scores['test_mae'][valid]), 
                    np.mean(scores['test_r2'][valid]), np.std(scores['test_r2'][valid])
                ])
            f.flush()
                        
    df_results_raw = pd.read_csv(checkpoint_csv)
    # 🌟 NEW: Filter out "ghost" combinations from previous runs if those features are currently turned off!
    valid_mask = df_results_raw['Features'].apply(lambda x: all(feat in available_groups for feat in str(x).split(" + ")))
    df_results = df_results_raw[valid_mask].sort_values(by="Spearman", ascending=False)
    if df_results.empty:
        print(f"⚠️ No valid completed combinations found for current settings. Skipping plotting.")
        return 
    df_results.to_excel(final_excel, index=False)
    
    # --- STAGE 3: Final Model ---
    print(f"\n🌟 STAGE 3: Extracting Model Rank #{rank_to_plot + 1} from leaderboard...")
    best_row = df_results.iloc[rank_to_plot]
    best_subs, best_feats = best_row['Subregions'].split(" + "), best_row['Features'].split(" + ")
    
    # Generate dynamic filename tags
    final_sub_tag = best_row['Subregions'].replace(' + ', '-')
    final_feat_tag = best_row['Features'].replace(' + ', '-')
    global_tag = f"{final_sub_tag}_{final_feat_tag}"
    
    # Extract the exact columns needed for the best model
    final_cols = filter_active_features(all_cols, best_subs, best_feats, global_features)
    
    # Clean the dataset for final evaluation
    cols_to_check = [target_col] + final_cols
    if weight_col and weight_col in df.columns: cols_to_check.append(weight_col)
    
    # Safely include the hue and oog columns without forcing drops if they are missing values
    df_subset_cols = cols_to_check + ([hue_col] if hue_col and hue_col in df.columns else []) + ([oog_col] if oog_col and oog_col in df.columns else [])
    final_df = df[df_subset_cols].replace([np.inf, -np.inf], np.nan).dropna(subset=cols_to_check).reset_index(drop=True)
    
    X_final = final_df[final_cols + ([weight_col] if weight_col else [])]
    y_final = final_df[target_col]
    hue_data_final = final_df[hue_col] if hue_col and hue_col in final_df.columns else None
    oog_data_final = final_df[oog_col] if oog_col and oog_col in final_df.columns else None
    
    # 🌟 NEW: Define the file path BEFORE training
    model_filename = os.path.join(output_dir, f"Production_{prefix}{model_name}_{target_col}_{global_tag}.joblib")
    
    # 🌟 NEW: Check if the model is already trained and saved!
    if os.path.exists(model_filename):
        print(f"\n⚡ Found existing Production Model! Loading '{model_filename}' (Skipping GridSearch)...")
        loaded_package = joblib.load(model_filename)
        if isinstance(loaded_package, dict):
            locked_best_estimator = loaded_package['model']
            
            # 🌟 CRITICAL FIX: Extract the EXACT feature order the model was trained on
            trained_features = loaded_package.get('features', final_cols)
            
            # 🌟 CRITICAL FIX: Force the X_final DataFrame to match this exact column order!
            expected_cols = trained_features + ([weight_col] if weight_col else [])
            X_final = X_final[expected_cols]
            
            # Update final_cols so the SHAP plot gets the correct names too
            final_cols = trained_features

            # 🌟 CRITICAL FIX: Load the hyperparameters so they appear on the plot!
            best_params = loaded_package.get('best_params', None)

        else:
            locked_best_estimator = loaded_package
            best_params = None

    else:
        print(f"\n⚙️ Training Final Production Model...")
        # EXACT REPLICATION: Build the final model exactly as it was tested during Grid Search!
        final_model = get_model(model_name, len(final_cols), transform_type, weight_col)
        
        # Fit once on the full dataset to extract the absolute best parameters for the plot text
        final_model.fit(X_final, y_final)
        best_params = final_model.best_params_ if hasattr(final_model, 'best_params_') else None
        
        # Extract the locked-in model so cross_val_predict doesn't run nested GridSearches
        locked_best_estimator = final_model.best_estimator_ if hasattr(final_model, 'best_estimator_') else final_model    
    # Generate Plots
    plot_best_model_diagnostics(
        X=X_final, 
        y=y_final, 
        subregions_name=best_row['Subregions'],
        features_name=best_row['Features'],
        model_name=model_name, 
        target_col=target_col, 
        output_dir=output_dir, 
        final_estimator=locked_best_estimator,
        best_params=best_params,
        feature_tag=global_tag,
        prefix=prefix,
        hue_data=hue_data_final,
        hue_name=hue_col
    )

    # 🌟 NEW: Optional Call for the Extended 9-Panel Diagnostic Plot!
    if extended_plots:
        plot_best_model_diagnostics_old(
            X=X_final, y=y_final,
            subregions_name=best_row['Subregions'], features_name=best_row['Features'],
            model_name=model_name, target_col=target_col, output_dir=output_dir,
            final_estimator=locked_best_estimator, best_params=best_params, feature_tag=global_tag,
            prefix=prefix, hue_data=hue_data_final, hue_name=hue_col, threshold=manual_threshold
        ) 
    
    # 🌟 NEW: Generate Out-of-Group plots if the custom split column was provided!
    if oog_data_final is not None:
        plot_out_of_group_diagnostics(
            X=X_final,
            y=y_final,
            group_labels=oog_data_final,
            subregions_name=best_row['Subregions'],
            features_name=best_row['Features'],
            model_name=model_name, 
            target_col=target_col, 
            output_dir=output_dir, 
            final_estimator=locked_best_estimator,
            feature_tag=global_tag,
            prefix=prefix,
            split_col_name=oog_col
        )

    generate_shap_analysis(
        model=locked_best_estimator, X=X_final, y=y_final, output_dir=output_dir, feature_names=final_cols, 
        model_name=model_name, target_col=target_col, prefix=prefix, feature_tag=global_tag, aaindex_desc=aaindex_desc
    )

    # 🌟 NEW: Only save if we didn't just load it from disk
    if not os.path.exists(model_filename):
        print(f"\n💾 Saving Final Production Model and Feature Metadata...")
        joblib.dump({
            'model': locked_best_estimator,
            'features': final_cols,
            'target': target_col,
            'best_params': best_params  # 🌟 CRITICAL FIX: Save the hyperparameters for the plot text!
        }, model_filename)
        print(f"✅ Production package successfully saved to: {model_filename}")
    else:
        print(f"\n✅ Production model already exists on disk. Skipping save.")

def evaluate_single_combination(df, target_col, model_name, output_dir, sub_combo, feat_combo, generated_features, 
                                transform_type=None, weight_col=None, prefix="targeted_", hue_col=None, oog_col=None, aaindex_desc=None):
    """Evaluates a single, specific combination of subregions and features."""
    print(f"\n==================================================================")
    print(f"🎯 TARGETED EVALUATION: {model_name} on {target_col}")
    print(f"   Regions: {sub_combo}")
    print(f"   Features: {feat_combo}")
    print(f"==================================================================")
    
    os.makedirs(output_dir, exist_ok=True)
    results_excel = os.path.join(output_dir, f"{prefix}single_eval_results_{model_name}_{target_col}.xlsx")
    
    all_cols = df.columns.tolist()
    global_features = [
        f for f in generated_features 
        if not f.startswith('seq_')
        and not f.startswith('Global_')
        and not f.startswith('CQA_')
        and not f.startswith('Propermab_')
    ]
    
    # Get active columns for this specific combination
    selected_cols = filter_active_features(all_cols, sub_combo, feat_combo, global_features)
    
    # 🌟 NEW: Always inject the Single Media feature into the selected columns
    if 'Media_Encoded' in all_cols and 'Media_Encoded' not in selected_cols:
        selected_cols.append('Media_Encoded')
        if 'Media' not in feat_combo:
            feat_combo = list(feat_combo) + ['Media']
            print(f"   -> 🧪 Auto-injected Media_Encoded feature into combination.")

    # Clean Data
    cols_to_check = [target_col] + selected_cols
    if weight_col and weight_col in df.columns: cols_to_check.append(weight_col)
    
    df_subset_cols = cols_to_check + ([hue_col] if hue_col and hue_col in df.columns else []) + ([oog_col] if oog_col and oog_col in df.columns else [])
    model_df = df[df_subset_cols].replace([np.inf, -np.inf], np.nan).dropna(subset=cols_to_check).reset_index(drop=True)
    
    if len(model_df) < 15:
        print("⚠️ Not enough data points to evaluate this combination!")
        return
        
    X = model_df[selected_cols + ([weight_col] if weight_col else [])]
    y = model_df[target_col]
    hue_data = model_df[hue_col] if hue_col and hue_col in model_df.columns else None
    oog_data = model_df[oog_col] if oog_col and oog_col in model_df.columns else None
    
    model = get_model(model_name, len(selected_cols), transform_type, weight_col)
    scorer = {'spearman': make_scorer(custom_spearman), 'r2': 'r2', 'rmse': 'neg_root_mean_squared_error', 'mae': 'neg_mean_absolute_error'}
    cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)
    
    scores = cross_validate(model, X, y, cv=cv, scoring=scorer, n_jobs=-1, error_score=np.nan)
    valid = ~np.isnan(scores['test_r2'])
    
    sub_name = " + ".join(sub_combo)
    feat_name = " + ".join(feat_combo)
    
    # 🌟 NEW: Move tag generation up here
    global_tag = f"{sub_name.replace(' + ', '-')}_{feat_name.replace(' + ', '-')}"
    model_filename = os.path.join(output_dir, f"Production_{prefix}{model_name}_{target_col}_{global_tag}.joblib")
    
    new_result = pd.DataFrame([{
        'Subregions': sub_name, 'Features': feat_name, 'Num_Features': len(selected_cols),
        'Spearman': np.mean(scores['test_spearman'][valid]), 'Spearman_Std': np.std(scores['test_spearman'][valid]),
        'RMSE': -np.mean(scores['test_rmse'][valid]), 'RMSE_Std': np.std(scores['test_rmse'][valid]),
        'MAE': -np.mean(scores['test_mae'][valid]), 'MAE_Std': np.std(scores['test_mae'][valid]),
        'R2': np.mean(scores['test_r2'][valid]), 'R2_Std': np.std(scores['test_r2'][valid])
    }])
    
    # Save to Excel
    if os.path.exists(results_excel):
        existing_df = pd.read_excel(results_excel)
        final_df = pd.concat([existing_df, new_result], ignore_index=True)
    else:
        final_df = new_result
    final_df.to_excel(results_excel, index=False)
    
    # 🌟 NEW: Check if the model is already trained and saved!
    if os.path.exists(model_filename):
        print(f"\n⚡ Found existing Production Model! Loading '{model_filename}' (Skipping GridSearch)...")
        loaded_package = joblib.load(model_filename)
        if isinstance(loaded_package, dict):
            locked_best_estimator = loaded_package['model']
            
            # 🌟 CRITICAL FIX: Extract the EXACT feature order the model was trained on
            trained_features = loaded_package.get('features', selected_cols)
            
            # 🌟 CRITICAL FIX: Force the X DataFrame to match this exact column order!
            expected_cols = trained_features + ([weight_col] if weight_col else [])
            X = X[expected_cols]
            
            # Update selected_cols so the SHAP plot gets the correct names too
            selected_cols = trained_features

            # 🌟 CRITICAL FIX: Load the hyperparameters so they appear on the plot!
            best_params = loaded_package.get('best_params', None)

        else:
            locked_best_estimator = loaded_package
            best_params = None

    else:
        print(f"\n⚙️ Training Final Production Model...")
        # Train full model for plots
        model.fit(X, y)
        best_params = model.best_params_ if hasattr(model, 'best_params_') else None
        locked_best_estimator = model.best_estimator_ if hasattr(model, 'best_estimator_') else model
    
    plot_best_model_diagnostics(
        X=X, y=y, subregions_name=sub_name, features_name=feat_name, model_name=model_name, 
        target_col=target_col, output_dir=output_dir, final_estimator=locked_best_estimator,
        best_params=best_params, feature_tag=global_tag, prefix=prefix, hue_data=hue_data, hue_name=hue_col
    )
    
    generate_shap_analysis(
        model=locked_best_estimator, X=X, y=y, output_dir=output_dir, feature_names=selected_cols, 
        model_name=model_name, target_col=target_col, prefix=prefix, feature_tag=global_tag, aaindex_desc=aaindex_desc
    )

    # 🌟 NEW: Only save if we didn't just load it from disk
    if not os.path.exists(model_filename):
        print(f"\n💾 Saving Final Production Model and Feature Metadata...")
        joblib.dump({
            'model': locked_best_estimator,
            'features': selected_cols,
            'target': target_col,
            'best_params': best_params  # 🌟 CRITICAL FIX: Save the hyperparameters for the plot text!
        }, model_filename)
        print(f"✅ Production package successfully saved to: {model_filename}")
    else:
        print(f"\n✅ Production model already exists on disk. Skipping save.")
    
    print(f"✅ Targeted Evaluation Complete! Results saved to {results_excel}")

def main():
    filepath = 'data/tubespin.csv'
    # When run SVR for ProA_HMW_ActiPro double check the top models in the excel. for CQA the respective model has a lowe rank in the file.
    targets_to_test = {'ELISA_Polyreactivity_Excell': 15.0}#, 'ProA_HMW_ActiPro':20, 'ProA_HMW_Excell': 20.0}

    # filepath = 'data/inhouse_supp_CD3+CD20only_UPDATED.csv'
    # targets_to_test = {
    #     'Purity%': 80.0,
    #     # 'HMW':10.0
    # }

    # filepath = 'data/2+1_Humanized_VH5-VL_anti-CD3_variant_sequece_GA.csv'
    # targets_to_test = {
    #     'Monomer': 80.0,
    #     'HMW%':10.0
    # }

    # filepath = 'data/tubespin_extended.csv'

    # targets_to_test = {
    #     # 'Monomer_combined':80.0,
    #     'HMW_combined':10.0
    # }
    models_to_test = ['SVR']#['XGBoost']#['ElasticNet', 'SVR','PLSRegression']#, 'SVR'] 
    
    transform_strategy = None
    # 🌟 NEW: Pass a list of models to extract both sets of features!
    esm_model_selections = ["facebook/esm2_t6_8M_UR50D", "facebook/esm2_t33_650M_UR50D"]
    antibody_format_column = 'Type'
    # antibody_format_column = 'Dataset'

    # 🌟 NEW: Type the exact name of your 0/1 split column here! 
    # If the column doesn't exist yet, it will just safely skip the plot.
    out_of_group_split_column = 'Manual_Split_Group'
    
    use_external_features = False
    use_cqa_features = False
    weighting_column = None
    DROP_MONOMER_OUTLIER = False 
    # 🌟 NEW: Set to True to generate the massive 9-panel legacy diagnostic and learning curve plot
    GENERATE_EXTENDED_PLOTS = False
    
    # 🌟 NEW: Targeted Single Evaluation Toggle
    RUN_SINGLE_EVAL = False
    single_eval_regions = ['Global_VH', 'Global_VL'] 
    single_eval_features = ['ESM_Big_650M_SVD50', 'Georgiev']

    # 🌟 NEW: Targeted Single Evaluation Toggle
    # 🌟 NEW: Media Type Integration
    USE_MEDIA_FEATURE = False
    media_column = 'Media_Type'

    try:
        df = load_and_clean_data(filepath, remove_outlier=DROP_MONOMER_OUTLIER)
        dataset_name = os.path.splitext(os.path.basename(filepath))[0]
        outlier_tag = "OutliersRemoved" if DROP_MONOMER_OUTLIER else "AllSamples"

        # 🌟 NEW: Process Media Feature (Single Integer Encoding)
        if USE_MEDIA_FEATURE and media_column in df.columns:
            print(f"\n🧬 Integrating '{media_column}' as a single universal integer feature...")

            # Extract mapping FIRST before overwriting
            media_mapping = dict(enumerate(df[media_column].astype('category').cat.categories))

            # Convert text (e.g. ActiPro, Excell) to integers (0, 1) IN-PLACE
            df[media_column] = df[media_column].astype('category').cat.codes

            print(f"   -> Added feature: '{media_column}' to ALL models.")
            print(f"   -> 📊 Media Dictionary: {media_mapping}")

        df_features, seq_cols, generated_features, aaindex_desc = extract_sequence_features(
            df, dataset_name=dataset_name, esm_model_names=esm_model_selections, cache_tag=outlier_tag
        )

        if use_external_features:
            ext_csv_path = os.path.join("feature_cache", dataset_name, f"{dataset_name}_propermab.csv")            
            if os.path.exists(ext_csv_path):
                ext_df = pd.read_csv(ext_csv_path)
                # Validation check to ensure data lengths match before merging
                if len(ext_df) == len(df_features): 
                    new_feats = [c for c in ext_df.columns if c not in df_features.columns]
                    df_features = pd.concat([df_features.reset_index(drop=True), ext_df[new_feats].reset_index(drop=True)], axis=1)
                    generated_features.extend(new_feats)
                    print(f"✅ SUCCESS: Loaded {len(new_feats)} external Propermab features.")
                else:
                    print(f"⚠️ WARNING: Propermab rows ({len(ext_df)}) do not match Dataset rows ({len(df_features)}). Skipping External Features!")
            else:
                print(f"⚠️ WARNING: 'use_external_features' is True, but file '{ext_csv_path}' was not found. Skipping!") 
        if seq_cols:
            for target_column, manual_threshold in targets_to_test.items():
                df_features_run = df_features.copy()
                generated_features_run = list(generated_features)
                
                if use_cqa_features:
                    media_type = "ActiPro" if "ActiPro" in target_column else "Excell"
                    cqa_npy_path = os.path.join("feature_cache", dataset_name, f"{dataset_name}_CQA_{media_type}.npy")
                    if os.path.exists(cqa_npy_path):
                        cqa_data = np.load(cqa_npy_path)
                        if cqa_data.ndim == 1: cqa_data = cqa_data.reshape(-1, 1)
                        # Strict validation to ensure the CQA matrix perfectly matches our current samples
                        if cqa_data.shape[0] == len(df_features_run): 
                            cqa_cols = [f"CQA_{media_type}_{i}" for i in range(cqa_data.shape[1])]
                            cqa_df = pd.DataFrame(cqa_data, columns=cqa_cols).reset_index(drop=True) 
                            df_features_run = pd.concat([df_features_run.reset_index(drop=True), cqa_df], axis=1)
                            generated_features_run.extend(cqa_cols)
                            print(f"✅ SUCCESS: Loaded {cqa_data.shape[1]} CQA features for {media_type}.")
                        else:
                            print(f"⚠️ WARNING: CQA array length ({cqa_data.shape[0]}) does not match Dataset rows ({len(df_features_run)}). Did you drop outliers? Skipping CQA features!")
                    else:
                        print(f"⚠️ WARNING: 'use_cqa_features' is True, but file '{cqa_npy_path}' was not found. Skipping CQA features!") 
                for model_name in models_to_test:
                    # Pass the single integer media feature down into the evaluation pools
                    if USE_MEDIA_FEATURE and media_column in df.columns:
                        if media_column not in generated_features_run:
                            generated_features_run.append(media_column)

                    if RUN_SINGLE_EVAL:
                        evaluate_single_combination(
                            df=df_features_run, target_col=target_column, model_name=model_name, output_dir=model_name,
                            sub_combo=single_eval_regions, feat_combo=single_eval_features,
                            generated_features=generated_features_run, transform_type=transform_strategy,
                            weight_col=weighting_column, prefix="targeted_", hue_col=antibody_format_column,
                            oog_col=out_of_group_split_column, aaindex_desc=aaindex_desc
                        )
                    else:
                        # print(f"\n==================================================================")
                        # print(f"🚀 EXHAUSTIVE SEARCH: Target = {target_column} | Model = {model_name}")
                        # print(f"==================================================================")
                        
                        # evaluate_exhaustive_combinations(
                        #     df_features_run, seq_cols, generated_features_run, target_col=target_column, 
                        #     model_name=model_name, output_dir=model_name, transform_type=transform_strategy, 
                        #     weight_col=weighting_column, hue_col=antibody_format_column, rank_to_plot=0,#, oog_col=out_of_group_split_column
                        #     aaindex_desc=aaindex_desc, extended_plots=GENERATE_EXTENDED_PLOTS, manual_threshold=manual_threshold
                        # )
                        
                        print(f"\n==================================================================")
                        print(f"🚀 EXHAUSTIVE SEARCH: GLOBAL SEQUENCES (VH, VL, Fv)")
                        print(f"==================================================================")
                        
                        evaluate_exhaustive_combinations(
                            df_features_run, ['Global_VH', 'Global_VL', 'Global_Fv'], generated_features_run, 
                            target_col=target_column, model_name=model_name, output_dir=model_name, 
                            transform_type=transform_strategy, weight_col=weighting_column, prefix="global_",
                            hue_col=antibody_format_column, rank_to_plot=3, oog_col=out_of_group_split_column,
                            aaindex_desc=aaindex_desc, extended_plots=GENERATE_EXTENDED_PLOTS, manual_threshold=manual_threshold
                        )
            
    except FileNotFoundError:
        print(f"Error: Could not find '{filepath}'.")

if __name__ == "__main__":
    main()

Loading data from data/tubespin.csv...

🧬 EVALUATING SUBREGION DIVERSITY & SUITABILITY
  -> ❌ DROPPED [CD20-VH-CH1_HCK]: Constant sequence across all samples (Zero Variance).
  -> ✅ KEPT [G4S Linker1_HCK]: Low variance (2 variants). Sequences: ['GGGGSGGGGSGGGGS', 'GGGGSGGGGS']
  -> ✅ KEPT [CD3 VH_HCK]: Low variance (3 variants). Sequences: ['EVQLVESGGGLVQPGGSLKLSCAASGFTFNKYAMNWVRQAPGKGLEWVARIRSKYNNYATYYADSVKDRFTISRDDSKNTAYLQMNNLKTEDTAVYYCVRHGNFGNSYISYWAYWGQGTLVTVSS', 'EVQLLESGGGVVRPGGSLRLSCAASGFTFNTYAMNWVRQAPGKGLEWVGRIRSKYNNYATYYADSVKDRFTISRDDSKNTLYLQMNSLKTEDTAVYYCTTHGNFGNSYVSWFAYWGQGTLVTVSS', 'EVQLVESGGGVVQPGRSLRLSCTASGFTFNTYAMNWVRQAPGKGLEWVGRIRSKYNNYATYYADSVKDRFTISRDDSKNTAYLQMNSLKTEDTAVYYCTRHGNFGNSYVSWFAYWGQGTLVTVSS']
  -> ❌ DROPPED [G4S Linker2_HCK]: Constant sequence across all samples (Zero Variance).
  -> ✅ KEPT [CD3 VL_HCK]: High variance (63 variants).
  -> ❌ DROPPED [HCK]: Constant sequence across all samples (Zero Variance).
  -> ✅ KEPT [Overall_HeavyChainKnob]: High variance